In [14]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path
import pandas as pd

In [15]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

vector_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "vector_retrieval_results.csv")
graph_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "graph_retrieval_results.csv")
hybrid_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "hybrid_retrieval_results.csv")

In [17]:
import ast
def parse_evidence(value):
    if pd.isna(value):
        return []
    
    if isinstance(value, list):
        return value
    
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return [value]

graph_results["evidence_parsed"] = graph_results["evidence"].apply(parse_evidence)

In [18]:
def get_graph_evidence(query_artist, retrieved_artist):
    evidence_rows = graph_results[
        (graph_results["query_artist"] == query_artist) &
        (graph_results["retrieved_artist"] == retrieved_artist)
    ]

    shared_members = []
    shared_tags = []

    for _, row in evidence_rows.iterrows():
        if row["evidence_type"] == "shared_member":
            shared_members.extend(row["evidence_parsed"])
        elif row["evidence_type"] == "shared_tag":
            shared_tags.extend(row["evidence_parsed"])

    return {
        "shared_members": shared_members,
        "shared_tags": shared_tags
    }

In [19]:
def generate_explanation(query_artist, retrieved_artist):
    
    vector_row = vector_results[
        (vector_results["query_artist"] == query_artist) &
        (vector_results["retrieved_artist"] == retrieved_artist)
    ]

    hybrid_row = hybrid_results[
        (hybrid_results["query_artist"] == query_artist) &
        (hybrid_results["retrieved_artist"] == retrieved_artist)
    ]

    evidence = get_graph_evidence(query_artist, retrieved_artist)

    explanation_parts = []
    explanation_parts.append(f"{retrieved_artist} was retrieved for {query_artist}.")
    
    if (vector_row.empty and hybrid_row.empty and 
        not evidence["shared_members"] and not evidence["shared_tags"]):
        
        explanation_parts.append(
            "This pair does not appear in the saved top-k vector, graph, or hybrid retrieval results for this query direction."
        )
        explanation_parts.append(
            "This does not necessarily mean the artists have no similarity or relationship; it only means the pair was not retrieved in the current saved top-k results."
        )
        return " ".join(explanation_parts)

    if not vector_row.empty:
        vector_score = vector_row.iloc[0]["similarity_score"]
        explanation_parts.append(
            f"In the vector retriever, this pair has a cosine similarity score of {vector_score:.3f}, based on artist embedding text."
        )

    if evidence["shared_members"]:
        members = ", ".join(evidence["shared_members"])
        explanation_parts.append(
            f"The graph retriever also found shared-member evidence: {members} connects {query_artist} and {retrieved_artist}."
        )

    if evidence["shared_tags"]:
        tags = ", ".join(evidence["shared_tags"][:5])
        explanation_parts.append(
            f"The graph also found shared tag evidence, including: {tags}."
        )

    if not hybrid_row.empty:
        hybrid_score = hybrid_row.iloc[0]["hybrid_score"]
        explanation_parts.append(
            f"The hybrid retriever combines vector similarity with graph evidence, giving this pair a hybrid score of {hybrid_score:.3f}."
        )

    if evidence["shared_members"]:
        explanation_parts.append(
            "This is a relationship-based discovery because the connection is supported by explicit historical graph evidence."
        )
    elif evidence["shared_tags"]:
        explanation_parts.append(
            "This is mainly a style or metadata-based discovery because the graph support comes from shared tags."
        )

    return " ".join(explanation_parts)

In [20]:
load_dotenv()

client = OpenAI(
  api_key=os.getenv("GROQ_API_KEY"),
  base_url="https://api.groq.com/openai/v1"
)

assert os.getenv("GROQ_API_KEY") is not None, "GROQ_API_KEY is missing"

In [10]:
# response = client.chat.completions.create(
#     model="llama-3.3-70b-versatile",
#     temperature=0.2,
#     max_tokens=100,
#     messages=[
#         {
#             "role": "user",
#             "content": "Rewrite this in one natural sentence: Pearl Jam and Soundgarden are connected through Matt Cameron."
#         }
#     ]
# )

# print(response.choices[0].message.content)

In [9]:
def rewrite_explanation(structured_explanation: str) -> str:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0.2,
        max_tokens=200,
        messages=[
            {
                "role": "system",
                "content": (
                    "You rewrite structured music-retrieval explanations into "
                    "natural, engaging prose. Use ONLY the facts given in the "
                    "input. Do not add any artist facts, relationships, or "
                    "claims not explicitly stated. Keep it to 2-3 sentences."
                )
            },
            {"role": "user", "content": structured_explanation}
        ]
    )
    return response.choices[0].message.content

In [21]:
template_output = generate_explanation("Pearl Jam", "Soundgarden")

print("TEMPLATE:")
print(template_output)
print()

print("LLM REWRITE:")
print(rewrite_explanation(template_output))

TEMPLATE:
Soundgarden was retrieved for Pearl Jam. In the vector retriever, this pair has a cosine similarity score of 0.741, based on artist embedding text. The graph retriever also found shared-member evidence: Matt Cameron connects Pearl Jam and Soundgarden. The graph also found shared tag evidence, including: alternative rock, grunge, hard rock, usa. The hybrid retriever combines vector similarity with graph evidence, giving this pair a hybrid score of 0.998. This is a relationship-based discovery because the connection is supported by explicit historical graph evidence.

LLM REWRITE:
Soundgarden was retrieved as a match for Pearl Jam, with a cosine similarity score of 0.741 based on artist embedding text. The connection between the two bands is also supported by shared-member evidence, specifically Matt Cameron, and shared tags like alternative rock and grunge. The combination of vector similarity and graph evidence gives this pair a hybrid score of 0.998, making it a relationship

In [ ]:
test_pairs = [
    ("Nirvana", "Soundgarden"),
    ("Deftones", "Soundgarden"),
    ("The Smashing Pumpkins", "Blur"),  
]

llm_results = []
for query, retrieved in test_pairs:
    template = generate_explanation(query, retrieved)
    llm_version = rewrite_explanation(template)
    llm_results.append({
        "query_artist": query,
        "retrieved_artist": retrieved,
        "template_explanation": template,
        "llm_explanation": llm_version
    })
    print(f"{query} -> {retrieved}")
    print("LLM:", llm_version)
    print()

Pearl Jam -> Soundgarden
LLM: Soundgarden was identified as a relevant match for Pearl Jam, with a cosine similarity score of 0.741 based on artist embedding text. The connection between the two bands is also supported by shared-member evidence, specifically Matt Cameron, and shared tags such as alternative rock and grunge. The combination of these factors resulted in a high hybrid score of 0.998, making this a relationship-based discovery.

Nirvana -> Soundgarden
LLM: Soundgarden was retrieved as a match for Nirvana, with a cosine similarity score of 0.712 based on artist embedding text. The connection between the two bands is also supported by shared-member evidence, specifically Jason Everman, and shared tags like alternative metal and grunge. The combination of vector similarity and graph evidence gives this pair a hybrid score of 0.998, making it a relationship-based discovery.

Deftones -> Soundgarden
LLM: The music retrieval system suggested Soundgarden for fans of Deftones, wit

In [23]:
pd.DataFrame(llm_results).to_csv(PROJECT_ROOT / "data" / "processed" / "llm_explanation_results.csv", index=False)

#### Conclusion — Phase 10: LLM Explanation Layer

This notebook adds a grounded natural-language rewriting layer on top of the
deterministic template explanations built in Phase 10B (notebook 15).

**Model used:** `llama-3.3-70b-versatile` via the Groq API (OpenAI-compatible
endpoint, free tier).

**Grounding constraint:** the system prompt explicitly instructs the model to
use *only* the facts present in the structured template input, and not to add
any artist facts, relationships, or claims not explicitly stated.

**Validation:** an early, unconstrained test call (no system prompt) produced
a rewrite that added unstated details ("drummer Matt Cameron", "iconic bands").
After adding the grounding constraint, four test pairs covering all major
evaluation categories from Phase 11 — two shared-member-driven matches
(Pearl Jam↔Soundgarden, Nirvana↔Soundgarden), one shared-tag-driven match
(Deftones→Soundgarden), and one no-change/agreement case
(The Smashing Pumpkins→Blur) — were rewritten with no added or fabricated
facts, confirming the constraint holds consistently rather than by chance.

**Important design note:** consistent with the project's core rule, the LLM
here only *explains* retrieval results already produced by the vector, graph,
and hybrid retrievers. It does not select, rank, or retrieve artists itself.

**Limitation:** unlike the deterministic template baseline, LLM output is not
fully reproducible — the same structured input can be phrased differently
across separate API calls. `temperature=0.2` was used to reduce (not
eliminate) this variability.

**Output saved to:** `data/processed/llm_explanation_results.csv`